<a href="https://colab.research.google.com/github/LuizaRamos/TOL506M_Final_Project/blob/main/notebooks/01_task1_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TÖL506M - Introduction to Deep Neural Network
## Final Project: Wildlife Image Classification
### Task 1 - Training from Scracht
**name:** Luiza V Sampaio Ramos, **email:** lvs2@gmail.com

**ALTER THE TEXT BELLOW**

This notebook

**Observation:** The first cell of code bellow was implemente to be able to run the notebook using Google Colab, while the rest o the code was initially wroten using DataSpell.

In [1]:
!rm -rf /content/TOL506M_Final_Project

In [2]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/LuizaRamos/TOL506M_Final_Project.git"
REPO_DIR = "/content/TOL506M_Final_Project"

# Always clone fresh in Colab
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

In [3]:
import sys
import os
import subprocess
import importlib
import json
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from collections import Counter
from PIL import Image
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torchvision import datasets, transforms
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

project_root = Path.cwd()
if project_root.name != "TOL506M_Final_Project":
    original = project_root
    while project_root.name != "TOL506M_Final_Project" and project_root != project_root.parent:
        project_root = project_root.parent

    if project_root.name == "TOL506M_Final_Project":
        os.chdir(project_root)
        print(f"Changed working directory from {original} to {project_root}")
    else:
        raise RuntimeError(
            "Could not locate the TOL506M_Final_Project root directory. "
            "Please run this notebook/script from within the project tree."
        )
else:
    print(f"Working directory: {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Plot styling
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 12})

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Working directory: /content/TOL506M_Final_Project

PyTorch version: 2.8.0+cu126
CUDA available: True


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alessiocorrado99/animals10")

print(f'Dataset downloaded: {path}\n')

Using Colab cache for faster access to the 'animals10' dataset.
Dataset downloaded: /kaggle/input/animals10



In [5]:
# Import project modules
from config import Config
from data.dataset import (WildlifeDataset, SplitIndices, stratified_split,
                          compute_class_counts, materialize_split, is_italian,
                          translate_names, get_class_names, get_data_loaders)
from data.augmentation import get_train_transforms, get_val_transforms
from models.resnet_scratch import ResNet18Scratch
from tasks.task1 import train_from_scratch
from utils.training import train_epoch, validate, EarlyStopping
from utils.evaluation import evaluate_model, get_confusion_matrix, compute_metrics
from utils.visualization import plot_training_curves, plot_confusion_matrix

data_fractions = Config.DATA_FRACTION
train_size = Config.TRAIN_SPLIT
val_size = Config.VAL_SPLIT
test_size = Config.TEST_SPLIT
random_seed = Config.RANDOM_SEED

data_path = Path(path) / 'raw-img'
Config.DATA_PATH = data_path

basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Download and split dataset as previously done on notebook 00_data_exploration
full_dataset = datasets.ImageFolder(root=str(data_path), transform=basic_transform)
dataset_transformed = translate_names(full_dataset)

wildlife = WildlifeDataset(str(data_path), transform=None)
train_idx_base, val_idx_fixed, test_idx_fixed = stratified_split(
    full_dataset,
    train_size=train_size,
    val_size=val_size,
    test_size=test_size,
    random_seed=random_seed
)

fixed = SplitIndices(train=train_idx_base, val=val_idx_fixed, test=test_idx_fixed)

print(f'Total images: {len(wildlife)}\n')

# Translating from Italian to English
dataset = translate_names(full_dataset)

data_fractions = Config.DATA_FRACTION

split_results = {}

for frac in data_fractions:
    train_loader, val_loader, test_loader, num_classes = get_data_loaders(
        data_path=str(data_path),
        batch_size=32,
        num_workers=0,
        train_split=train_size,
        val_split=val_size,
        test_split=test_size,
        use_augmentation=False,
        random_seed=random_seed,
        data_fraction=frac,
        save_processed_root="data/processed",
        fixed_indices=fixed
    )

Total images: 26179



In [6]:
# Create output dirs if they don't exist
Config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
Config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
Config.METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Splits:", train_size, val_size, test_size)
print("Random seed:", random_seed)

Splits: 0.7 0.15 0.15
Random seed: 7278


In [7]:
# Task 1: Train ResNet-18 from scratch for each data fraction

all_results = []

for frac in data_fractions:
  print(f"Starting training for data fraction = {frac*100:.0f}%\n")

  result = train_from_scratch(
        config=Config,
        data_fraction=frac,
        version_RestNet=18,
        save_model=True,
        fixed_indices=fixed
  )

  all_results.append(result)

print("\nFinished training for all fractions.")

Starting training for data fraction = 10%

Training on 10 classes.
Training samples: 58.
Validation samples: 123.
Test samples: 123.
Model ResNet-18 from scratch.
Parameters: 11,181,642


Validation: 100%|██████████| 123/123 [00:05<00:00, 23.57it/s]



 Epoch 1/100:
 Train Loss: 2.2201, Train Acc: 19.00%
 Val Loss: 2.1647, Val Acc: 22.74%
Saved best model (Val Acc: 22.74%) to /content/TOL506M_Final_Project/results/models/task1_scratch_best_fraction_0.10.pth.


Validation: 100%|██████████| 123/123 [00:05<00:00, 24.51it/s]



 Epoch 2/100:
 Train Loss: 2.1646, Train Acc: 22.05%
 Val Loss: 2.1268, Val Acc: 22.38%


...

Validation: 100%|██████████| 123/123 [00:04<00:00, 25.09it/s]



 Epoch 99/100:
 Train Loss: 0.8534, Train Acc: 69.16%
 Val Loss: 1.2833, Val Acc: 58.52%


Validation: 100%|██████████| 123/123 [00:04<00:00, 25.19it/s]


 Epoch 100/100:
 Train Loss: 0.8862, Train Acc: 69.38%
 Val Loss: 1.2925, Val Acc: 58.65%
Training completed in 1072.09 seconds.


ValueError: unknown is not supported

In [ ]:
# Save combined summary JSON
summary_path = Config.METRICS_DIR / "task1_scratch_all_fractions_summary.json"

with open(summary_path, "w") as f:
    json.dump(all_results, f, indent=4)

print(f"Saved combined summary to {summary_path}")

In [ ]:
from IPython.display import Image, display

for frac in data_fractions:
    plot_path = Config.PLOTS_DIR / f"task1_scratch_learning_curves_{frac:.2f}.png"
    if plot_path.exists():
        print(f"\nLearning curves for fraction = {frac*100:.0f}%")
        display(Image(filename=str(plot_path)))
    else:
        print(f"Plot not found for fraction {frac:.2f}: {plot_path}")